# ML-02 — Research Question and Provisional Lane

**FlyRank AI Internship · Week 1 · Draft for Kaggle**

This is a provisional research plan, not a finished ML model. I will review the code outputs before committing the notebook. The lane can be changed until the end of Week 4.

**Data:** FlyRank anonymized starter CSV (`data/raw/content_refresh_anonymized.csv`).

## 1. My lane (or freestyle) and why

**Chosen lane: Lane 2 — Refresh / Content Opportunity Scoring.**

I want to investigate which content pages should be reviewed first when a team has limited time to improve search performance. This lane interests me because it connects measurable signals such as impressions, content freshness, search position, and traffic movement to practical decisions. In the first two starter notebooks, I learned how a hand-written ranking rule can be compared with a simple machine-learning model, and why a high score must be checked for leakage. I now want to develop a transparent page-review priority system and compare it fairly with a simple baseline. I am choosing this direction provisionally and may refine it after studying the larger warehouse data.

## 2. The question: decision, action, cost of a wrong call

**Research question.** Among pages with enough measurable search activity, which ones should a content or SEO team **review first** for updating, expanding, protecting, consolidating, or monitoring?

**Unit of analysis.** One pseudonymized content page (`content_id`) in the starter dataset.

**Who makes the decision and what do they do?** An SEO analyst or content editor reviews a ranked queue of pages. The reviewer checks the evidence and decides whether to refresh or expand a page, improve its search snippet, protect strong content, consolidate redundant content, or monitor it without making a change.

**Output.** A ranked list of page IDs with an opportunity score, an interpretable reason code (for example, *declining with demand* or *stale but visible*), a suggested **review** action, and an uncertainty/confidence note. No real client identities or URLs will be displayed.

**Cost of a wrong recommendation.** A false positive could waste an editor's time on a page that does not need attention, or encourage a harmful change to a well-performing page. A false negative could leave a genuinely declining or high-opportunity page unreviewed. Therefore, the system should assist a human rather than automatically edit content.

**Why data or ML could help.** Thousands of pages can have competing signals: a page might have many impressions but weak CTR, or be old without declining. A clear rule-based score is a necessary baseline. If a validated model can rank review candidates better using several independent signals, ML may help allocate limited review time more effectively. Complexity is not useful unless it beats the simple rule on appropriate held-out data.

**Initial task and success measure.** Ranking/scoring. On the starter data, compare baseline and model **Precision@50** for the existing *current-window decline proxy*, clearly labeled as a teaching proxy rather than evidence of future benefits. For the eventual capstone, prefer a future-window observed outcome where the warehouse history supports it, with leakage-safe and client-aware validation.

## 3. Quick look at the data (2–3 real numbers)

The following code loads the actual FlyRank starter dataset and prints **three measured numbers**: (1) pages, (2) pages declining with at least 100 impressions, and (3) stale pages with at least 500 impressions. These are candidates for *review*, not pages proven to improve after a refresh.

I will verify the outputs before submitting; no counts are guessed in advance.

In [1]:
# Kaggle-ready setup: reuse or clone the public starter repository, then load its CSV.
from pathlib import Path
import os
import subprocess
import pandas as pd
from IPython.display import display, Markdown

REPO_URL = "https://github.com/rahadul231/flyrank-ml-internship-starter.git"
repo_root = (Path("/kaggle/working/flyrank-ml-internship-starter")
             if Path("/kaggle/working").is_dir()
             else Path.cwd())

if Path("/kaggle/working").is_dir() and not (repo_root / ".git").exists():
    if repo_root.exists():
        raise RuntimeError(f"Folder exists but isn't a Git repository: {repo_root}")
    print("Cloning public FlyRank starter repository...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo_root)], check=True)

csv_path = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"
if not csv_path.is_file():
    raise FileNotFoundError(f"Starter CSV not found: {csv_path}")

df = pd.read_csv(csv_path)
required = ["content_id", "impressions_90d", "trend_direction", "days_since_last_update"]
missing = [c for c in required if c not in df.columns]
assert not missing, f"Missing required columns: {missing}"

impressions = pd.to_numeric(df["impressions_90d"], errors="coerce")
update_age = pd.to_numeric(df["days_since_last_update"], errors="coerce")

# These rules are transparent *review flags*, not proven causal benefits.
declining_with_demand = (df["trend_direction"].eq("down") & (impressions >= 100))
stale_visible = ((update_age >= 180) & (impressions >= 500))

number_pages = len(df)
number_declining = int(declining_with_demand.sum())
number_stale_visible = int(stale_visible.sum())

print("Starter dataset observations (from this run):")
print(f"1. Content pages: {number_pages:,}")
print(f"2. Declining pages with >=100 impressions: {number_declining:,} ({number_declining / number_pages:.1%})")
print(f"3. Stale pages with >=500 impressions: {number_stale_visible:,} ({number_stale_visible / number_pages:.1%})")
print("Note: these categories may overlap.")

display(Markdown(
    f"**Evidence for my lane.** The starter dataset contains **{number_pages:,} pages**. "
    f"Under two transparent screening rules, **{number_declining:,}** show a decline "
    f"with at least 100 impressions and **{number_stale_visible:,}** are stale "
    f"yet have at least 500 impressions. These measured candidate counts justify "
    "testing a review-priority queue; they do not demonstrate that editing these pages will increase traffic."
))

Cloning public FlyRank starter repository...


Cloning into '/kaggle/working/flyrank-ml-internship-starter'...


Starter dataset observations (from this run):
1. Content pages: 30,000
2. Declining pages with >=100 impressions: 13,152 (43.8%)
3. Stale pages with >=500 impressions: 17 (0.1%)
Note: these categories may overlap.


**Evidence for my lane.** The starter dataset contains **30,000 pages**. Under two transparent screening rules, **13,152** show a decline with at least 100 impressions and **17** are stale yet have at least 500 impressions. These measured candidate counts justify testing a review-priority queue; they do not demonstrate that editing these pages will increase traffic.

## 4. Careful words: what I can and can't claim

**What I can claim after running the analysis:** The anonymized starter dataset contains observable signals for grouping and prioritizing pages. I can measure the size of possible review groups and test whether a transparent scoring rule or a model ranks pages better on a stated **proxy** label. I can give analysts a reasoned, review-first ranking and discuss trade-offs and uncertainty.

**What I cannot claim:** I cannot claim to have discovered Google's ranking algorithm, that a page is guaranteed to recover after being edited, or that an observed relationship is causal. I also cannot present the starter's `trend_direction == "down"` label as a future outcome or feed `trend_direction`/`trend_pct` into a model intended to predict that label: that would leak the answer. I will not expose real client names, queries, page URLs, or private identifiers.

**Next research steps:** Validate the screening rules and metric definitions; test a simple baseline; investigate whether the warehouse data supports a later outcome window; and evaluate by clients or time periods to avoid misleading scores. Editors, not the model, make final publication decisions.

## 5. Self-check

After running **Run All** in Kaggle, confirm:

- [ ] The provisional lane is stated and I agree with it.
- [ ] The decision, person, action, unit, output, error cost, and role of ML are clear.
- [ ] Section 3 prints three **real** numbers from the starter CSV without errors.
- [ ] Claims are observational/decision-support, not causal or guarantees of recovery.
- [ ] No private names, client details, domains, URLs, or queries appear in outputs.
- [ ] Save a fully executed `.ipynb` and upload it to **`work/notebooks/w01_research_question.ipynb`** in my GitHub repository, replacing the existing skeleton.
- [ ] Submit the **repository URL**, not this individual notebook link, for ML-02.